# NovaSuite — Revenue Analysis


---

## Context



The goal of this section is to answer six core business questions using only **Pandas and NumPy**.  


| # | Business Question |
|---|-------------------|
| 1 | Is revenue growing, declining, or stagnant over time? |
| 2 | Which plan drives the most revenue? |
| 3 | Which industries generate the most value? |
| 4 | Is discounting reducing realized revenue? |
| 5 | Are upsell opportunities being captured effectively? |
| 6 | Which customers generate the highest average revenue per account? |

**Key metrics tracked:** MRR · ARPA · Expansion Revenue · Discount Leakage · Plan Revenue Mix

---

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

In [ ]:
# ── Step 1: Load each CSV file into its own dataframe ─────────────────────────
customers = pd.read_csv('customers.csv')
revenue   = pd.read_csv('revenue.csv')
plans     = pd.read_csv('plans.csv')
usage     = pd.read_csv('usage.csv')
churn     = pd.read_csv('churn.csv')

print('Files loaded:')
print(f'  customers : {customers.shape[0]} rows, {customers.shape[1]} columns')
print(f'  revenue   : {revenue.shape[0]} rows, {revenue.shape[1]} columns')
print(f'  plans     : {plans.shape[0]} rows, {plans.shape[1]} columns')
print(f'  usage     : {usage.shape[0]} rows, {usage.shape[1]} columns')
print(f'  churn     : {churn.shape[0]} rows, {churn.shape[1]} columns')

In [ ]:
# ── Step 2: Convert date columns from text to proper datetime format ───────────
# pd.to_datetime() tells Pandas to treat these columns as dates, not plain text.
# This lets us later sort by month, extract year, etc.

revenue['month']         = pd.to_datetime(revenue['month'])
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
churn['churn_date']      = pd.to_datetime(churn['churn_date'])
usage['month']           = pd.to_datetime(usage['month'])

print('Date columns parsed successfully.')

In [ ]:
# ── Step 3: Build the master dataframe ────────────────────────────────────────
# We start with revenue (one row per customer per month) and add
# customer attributes (industry, company_size, etc.) and plan details.
#
# .merge() joins two dataframes on a shared column — similar to a SQL JOIN.
# how='left' means: keep all rows from the left table even if there is no match.

# Join customer attributes onto revenue
master = revenue.merge(
    customers[['customer_id', 'industry', 'company_size', 'country', 'signup_date']],
    on='customer_id',
    how='left'
)

# Join plan details (plan name, list price, max users)
master = master.merge(
    plans[['plan_id', 'plan_name', 'monthly_price', 'max_users']],
    on='plan_id',
    how='left'
)

print('master dataframe built.')
print(f'  Shape      : {master.shape[0]} rows × {master.shape[1]} columns')
print(f'  Customers  : {master["customer_id"].nunique()} unique')
print(f'  Date range : {master["month"].min().date()} to {master["month"].max().date()}')
print(f'  Columns    : {list(master.columns)}')

In [ ]:
# ── Step 4: Preview the master dataframe ─────────────────────────────────────
# Each row = one customer in one month.
# This is the base table we will use for all analysis below.

master.head(5)

---

## 1. Monthly Recurring Revenue (MRR) Trend

**Business question:** Is revenue growing, declining, or stagnant over time?

**What is MRR?**  
MRR (Monthly Recurring Revenue) is the total revenue collected from all customers in a given month.  
It is the most important top-line metric for a SaaS business — it tells you whether the business is growing.

**Which column do we use?**  
We use `subscription_revenue` — this is the net amount actually collected after discounts are applied.  
It reflects real cash received, not the list price.

**What we will calculate:**
- Total MRR per month (grouped summary table)
- Highest and lowest revenue months
- Year-over-year comparison using annual totals
- Quarter-by-quarter trend for the most recent year

In [ ]:
# ── 1a: Group revenue by month ────────────────────────────────────────────────
# groupby('month') splits the data into one group per month.
# .agg() then calculates multiple summaries at once for each group.
# 'sum' adds up all values; 'nunique' counts how many unique values exist.

monthly_mrr = master.groupby('month').agg(
    mrr               = ('subscription_revenue', 'sum'),
    expansion_revenue = ('expansion_revenue',    'sum'),
    total_discount    = ('discount',             'sum'),
    active_customers  = ('customer_id',          'nunique')
).reset_index()

# Format the month column to a readable label (e.g. "Jan 2023")
monthly_mrr['month_label'] = monthly_mrr['month'].dt.strftime('%b %Y')

print(f'Monthly MRR table built: {len(monthly_mrr)} months of data.')

In [ ]:
# ── 1b: Key MRR statistics ────────────────────────────────────────────────────
# .mean()  → average value across all months
# .idxmax() → the row index of the maximum value (used to look up that row)
# .idxmin() → the row index of the minimum value

avg_mrr          = monthly_mrr['mrr'].mean()
highest_idx      = monthly_mrr['mrr'].idxmax()
lowest_idx       = monthly_mrr['mrr'].idxmin()
highest_mrr_row  = monthly_mrr.loc[highest_idx]
lowest_mrr_row   = monthly_mrr.loc[lowest_idx]

print('=' * 52)
print('  MRR OVERVIEW')
print('=' * 52)
print(f'  Total months tracked  : {len(monthly_mrr)}')
print(f'  Average monthly MRR   : ${avg_mrr:>10,.2f}')
print(f'  Highest MRR month     : {highest_mrr_row["month_label"]:<10}  ${highest_mrr_row["mrr"]:>10,.2f}')
print(f'  Lowest  MRR month     : {lowest_mrr_row["month_label"]:<10}  ${lowest_mrr_row["mrr"]:>10,.2f}')
print('=' * 52)

In [ ]:
# ── 1c: MRR table — last 12 months ───────────────────────────────────────────
# .tail(12) selects the last 12 rows (most recent 12 months).
# .copy() creates a separate copy so we can safely rename columns
# without affecting the original monthly_mrr dataframe.

last_12 = monthly_mrr.tail(12).copy()
last_12 = last_12[['month_label', 'mrr', 'active_customers', 'expansion_revenue', 'total_discount']]
last_12.columns = ['Month', 'MRR ($)', 'Active Customers', 'Expansion Rev ($)', 'Total Discount ($)']

# Round numeric columns to 2 decimal places for clean display
last_12['MRR ($)']            = last_12['MRR ($)'].round(2)
last_12['Expansion Rev ($)']  = last_12['Expansion Rev ($)'].round(2)
last_12['Total Discount ($)'] = last_12['Total Discount ($)'].round(2)
last_12 = last_12.reset_index(drop=True)

print('MRR — Last 12 Months:')
last_12

In [ ]:
# ── 1d: Annual MRR comparison ─────────────────────────────────────────────────
# We extract the year from the month column using .dt.year
# Then group by year to get annual totals and compare growth.

master['year'] = master['month'].dt.year

annual_mrr = master.groupby('year').agg(
    total_mrr        = ('subscription_revenue', 'sum'),
    active_customers = ('customer_id',          'nunique'),
    months_active    = ('month',                'nunique')
).reset_index()

annual_mrr['avg_monthly_mrr'] = (annual_mrr['total_mrr'] / annual_mrr['months_active']).round(2)

# Year-over-year growth: compare each year to the previous year.
# .shift(1) moves the column down by one row so each year aligns with the prior year.
annual_mrr['prev_year_mrr'] = annual_mrr['total_mrr'].shift(1)
annual_mrr['yoy_growth_pct'] = (
    (annual_mrr['total_mrr'] - annual_mrr['prev_year_mrr'])
    / annual_mrr['prev_year_mrr']
    * 100
).round(1)

display_annual = annual_mrr[['year', 'total_mrr', 'avg_monthly_mrr', 'active_customers', 'yoy_growth_pct']].copy()
display_annual.columns = ['Year', 'Total MRR ($)', 'Avg Monthly MRR ($)', 'Unique Customers', 'YoY Growth (%)']
display_annual['Total MRR ($)']       = display_annual['Total MRR ($)'].round(2)
display_annual['Avg Monthly MRR ($)'] = display_annual['Avg Monthly MRR ($)'].round(2)
display_annual = display_annual.reset_index(drop=True)

print('Annual MRR Summary:')
display_annual

In [ ]:
# ── 1e: Quarterly MRR for the most recent full year ───────────────────────────
# .dt.quarter extracts the quarter number (1, 2, 3, or 4) from a date column.
# We filter to 2023 only — the most recent full calendar year in the dataset.

master['quarter'] = master['month'].dt.quarter

quarterly_2023 = master[master['year'] == 2023].groupby('quarter').agg(
    total_mrr       = ('subscription_revenue', 'sum'),
    num_customers   = ('customer_id',          'nunique')
).reset_index()

quarterly_2023['quarter_label'] = 'Q' + quarterly_2023['quarter'].astype(str) + ' 2023'
quarterly_2023['avg_mrr_per_customer'] = (
    quarterly_2023['total_mrr'] / quarterly_2023['num_customers']
).round(2)

display_q = quarterly_2023[['quarter_label', 'total_mrr', 'num_customers', 'avg_mrr_per_customer']].copy()
display_q.columns = ['Quarter', 'Total MRR ($)', 'Active Customers', 'Avg MRR / Customer ($)']
display_q['Total MRR ($)'] = display_q['Total MRR ($)'].round(2)
display_q = display_q.reset_index(drop=True)

print('Quarterly MRR — 2023:')
display_q

**What does this tell us?**

- The annual MRR summary shows whether revenue has grown each year since NovaSuite launched in 2021.
- The YoY growth column makes it easy to spot accelerating or slowing growth.
- The quarterly breakdown for 2023 shows whether revenue is front-loaded (Q1/Q2 heavy) or back-loaded (Q3/Q4 heavy).

---

## 2. Revenue by Plan

**Business question:** Which plan drives the most revenue?



**What we will calculate:**
- Total revenue and customer count per plan
- Each plan's percentage share of total revenue
- Average revenue per customer per plan
- Revenue efficiency: how much revenue each plan generates relative to its max user capacity

In [ ]:
# ── 2a: Group revenue by plan ─────────────────────────────────────────────────
plan_revenue = master.groupby('plan_name').agg(
    total_revenue    = ('subscription_revenue', 'sum'),
    unique_customers = ('customer_id',          'nunique'),
    total_months     = ('month',                'count'),
    list_price       = ('monthly_price',        'mean')
).reset_index()

# Percentage of total revenue contributed by each plan
grand_total = plan_revenue['total_revenue'].sum()
plan_revenue['pct_of_total'] = (plan_revenue['total_revenue'] / grand_total * 100).round(1)

# Average cumulative revenue per customer (total rev / unique customers)
plan_revenue['avg_rev_per_customer'] = (plan_revenue['total_revenue'] / plan_revenue['unique_customers']).round(2)

# Sort from highest to lowest total revenue
plan_revenue = plan_revenue.sort_values('total_revenue', ascending=False).reset_index(drop=True)

display_plan = plan_revenue[['plan_name', 'unique_customers', 'list_price', 'total_revenue', 'pct_of_total', 'avg_rev_per_customer']].copy()
display_plan.columns = ['Plan', 'Customers', 'List Price ($)', 'Total Revenue ($)', '% of Total', 'Avg Rev / Customer ($)']
display_plan['Total Revenue ($)']       = display_plan['Total Revenue ($)'].round(2)
display_plan['Avg Rev / Customer ($)']  = display_plan['Avg Rev / Customer ($)'].round(2)

print('Revenue by Plan (ranked highest to lowest):')
display_plan

In [ ]:
# ── 2b: Revenue efficiency — revenue per max user slot ────────────────────────
# "Revenue efficiency" measures how much revenue each plan generates
# relative to the maximum number of users it allows.
# A low number means the plan has large user capacity but generates little revenue per slot.

efficiency = master.groupby(['plan_name', 'max_users']).agg(
    total_revenue = ('subscription_revenue', 'sum'),
    customers     = ('customer_id',          'nunique')
).reset_index()

# Total potential capacity = customers × max_users
efficiency['total_user_capacity']    = efficiency['customers'] * efficiency['max_users']
efficiency['revenue_per_user_slot']  = (efficiency['total_revenue'] / efficiency['total_user_capacity']).round(2)
efficiency = efficiency.sort_values('revenue_per_user_slot', ascending=False).reset_index(drop=True)

display_eff = efficiency[['plan_name', 'max_users', 'customers', 'total_revenue', 'revenue_per_user_slot']].copy()
display_eff.columns = ['Plan', 'Max Users', 'Customers', 'Total Revenue ($)', 'Revenue per User Slot ($)']
display_eff['Total Revenue ($)'] = display_eff['Total Revenue ($)'].round(2)

print('Revenue Efficiency by Plan (revenue per max user slot):')
print('  → A higher number means the plan generates more revenue per unit of user capacity.')
print()
display_eff

In [ ]:
# ── 2c: Plan revenue summary printout ─────────────────────────────────────────
print('=' * 55)
print('  PLAN REVENUE SUMMARY')
print('=' * 55)
for i in range(len(plan_revenue)):
    row = plan_revenue.iloc[i]
    print(f"  {row['plan_name']:<12} | ${row['total_revenue']:>10,.2f} | {row['pct_of_total']:>5.1f}% of total")
print('  ─' * 27)
print(f"  {'TOTAL':<12} | ${grand_total:>10,.2f} | 100.0% of total")
print('=' * 55)

**What does this tell us?**

- Revenue is not evenly distributed — higher-tier plans punch above their weight relative to their customer count.
- The percentage contribution column shows which plan is carrying the most revenue, independent of customer volume.
- Revenue per user slot shows which plan is most and least efficient — a low value flags wasted capacity.

---

## 3. Revenue by Industry

**Business question:** Which industries generate the most value?

NovaSuite serves 8 industries: FinTech, Healthcare, SaaS/Tech, EdTech, E-Commerce, Logistics, Retail, Manufacturing.

Understanding which verticals are most valuable helps prioritise:
- Where to invest in sales and marketing
- Which industries deserve dedicated pricing or packaging
- Which segments are under-monetised relative to their size

**What we will calculate:**
- Total revenue per industry
- Customer count and average revenue per customer
- Share of total revenue by industry
- Revenue rank vs customer rank (to find over- and under-performing segments)

In [ ]:
# ── 3a: Group revenue by industry ─────────────────────────────────────────────
industry_revenue = master.groupby('industry').agg(
    total_revenue    = ('subscription_revenue', 'sum'),
    unique_customers = ('customer_id',          'nunique'),
    total_discount   = ('discount',             'sum')
).reset_index()

industry_revenue['avg_rev_per_customer'] = (
    industry_revenue['total_revenue'] / industry_revenue['unique_customers']
).round(2)

industry_revenue['pct_of_total'] = (
    industry_revenue['total_revenue'] / industry_revenue['total_revenue'].sum() * 100
).round(1)

# Sort by total revenue (highest first)
industry_revenue = industry_revenue.sort_values('total_revenue', ascending=False).reset_index(drop=True)

# Add rank columns for comparison
industry_revenue['revenue_rank']  = industry_revenue['total_revenue'].rank(ascending=False).astype(int)
industry_revenue['customer_rank'] = industry_revenue['unique_customers'].rank(ascending=False).astype(int)

display_ind = industry_revenue[['industry', 'unique_customers', 'total_revenue', 'avg_rev_per_customer', 'pct_of_total', 'revenue_rank', 'customer_rank']].copy()
display_ind.columns = ['Industry', 'Customers', 'Total Revenue ($)', 'Avg Rev / Customer ($)', '% of Total', 'Rev Rank', 'Cust Rank']
display_ind['Total Revenue ($)']       = display_ind['Total Revenue ($)'].round(2)
display_ind['Avg Rev / Customer ($)']  = display_ind['Avg Rev / Customer ($)'].round(2)

print('Revenue by Industry (ranked by total revenue):')
display_ind

In [ ]:
# ── 3b: Rank mismatch analysis ────────────────────────────────────────────────
# A positive rank difference means a segment has MORE customers than its revenue rank suggests.
# These are segments with a large customer base but low per-customer revenue — under-monetised.

industry_revenue['rank_diff'] = industry_revenue['customer_rank'] - industry_revenue['revenue_rank']

print('Industries with more customers than revenue (under-monetised):')
under = industry_revenue[industry_revenue['rank_diff'] > 0][['industry', 'unique_customers', 'avg_rev_per_customer', 'rank_diff']].copy()
under = under.sort_values('rank_diff', ascending=False).reset_index(drop=True)
under.columns = ['Industry', 'Customers', 'Avg Rev / Customer ($)', 'Customer Rank Ahead of Revenue Rank']
under['Avg Rev / Customer ($)'] = under['Avg Rev / Customer ($)'].round(2)
print(under)

print()
print('Industries generating more revenue than their customer count suggests (over-performing):')
over = industry_revenue[industry_revenue['rank_diff'] < 0][['industry', 'unique_customers', 'avg_rev_per_customer', 'rank_diff']].copy()
over = over.sort_values('rank_diff').reset_index(drop=True)
over.columns = ['Industry', 'Customers', 'Avg Rev / Customer ($)', 'Revenue Rank Ahead of Customer Rank']
over['Revenue Rank Ahead of Customer Rank'] = over['Revenue Rank Ahead of Customer Rank'].abs()
over['Avg Rev / Customer ($)'] = over['Avg Rev / Customer ($)'].round(2)
print(over)

In [ ]:
# ── 3c: Top and bottom industry quick summary ─────────────────────────────────
top_industry    = industry_revenue.iloc[0]
bottom_industry = industry_revenue.iloc[-1]
highest_arpa_row = industry_revenue.sort_values('avg_rev_per_customer', ascending=False).iloc[0]
lowest_arpa_row  = industry_revenue.sort_values('avg_rev_per_customer').iloc[0]

print('=' * 58)
print('  INDUSTRY REVENUE HIGHLIGHTS')
print('=' * 58)
print(f"  Highest total revenue : {top_industry['industry']:<15} ${top_industry['total_revenue']:>10,.2f}")
print(f"  Lowest  total revenue : {bottom_industry['industry']:<15} ${bottom_industry['total_revenue']:>10,.2f}")
print(f"  Highest avg rev/cust  : {highest_arpa_row['industry']:<15} ${highest_arpa_row['avg_rev_per_customer']:>10,.2f}")
print(f"  Lowest  avg rev/cust  : {lowest_arpa_row['industry']:<15} ${lowest_arpa_row['avg_rev_per_customer']:>10,.2f}")
print('=' * 58)

**What does this tell us?**

- Comparing revenue rank vs customer rank reveals which industries punch above or below their weight.
- An industry that ranks high in customers but low in revenue is under-monetised — it has volume but low value per account.
- An industry that ranks high in revenue despite fewer customers is a high-value segment worth protecting and growing.

---

## 4. Discount Analysis

**Business question:** Is discounting reducing realized revenue?



A discount rate is the discount as a percentage of the list price:  
`discount_rate = (discount / base_revenue) × 100`

**What we will calculate:**
- Overall discount rate across the whole business
- Total revenue lost to discounts (discount leakage)
- Discount rate by industry (which verticals receive the most discounts)
- Discount rate by plan (which plans are most discounted)

In [ ]:
# ── 4a: Overall discount summary ─────────────────────────────────────────────
total_list_rev    = master['base_revenue'].sum()
total_discounts   = master['discount'].sum()
total_net_rev     = master['subscription_revenue'].sum()
overall_disc_rate = (total_discounts / total_list_rev * 100).round(2)

print('=' * 58)
print('  DISCOUNT LEAKAGE — OVERALL')
print('=' * 58)
print(f"  Total list revenue (before discounts) : ${total_list_rev:>12,.2f}")
print(f"  Total discounts given                 : ${total_discounts:>12,.2f}")
print(f"  Total net revenue (after discounts)   : ${total_net_rev:>12,.2f}")
print(f"  Overall discount rate                 : {overall_disc_rate:>11.2f}%")
print('=' * 58)
print()
print(f'  → For every $100 of list price charged, only ${100 - overall_disc_rate:.2f} was collected.')

In [ ]:
# ── 4b: Discount by industry ──────────────────────────────────────────────────
disc_by_industry = master.groupby('industry').agg(
    list_revenue  = ('base_revenue',             'sum'),
    total_disc    = ('discount',                 'sum'),
    net_revenue   = ('subscription_revenue',     'sum')
).reset_index()

disc_by_industry['discount_rate_pct']    = (disc_by_industry['total_disc'] / disc_by_industry['list_revenue'] * 100).round(2)
disc_by_industry['revenue_lost']         = disc_by_industry['total_disc'].round(2)

# Sort: highest discount rate first
disc_by_industry = disc_by_industry.sort_values('discount_rate_pct', ascending=False).reset_index(drop=True)

display_disc_ind = disc_by_industry[['industry', 'list_revenue', 'total_disc', 'net_revenue', 'discount_rate_pct']].copy()
display_disc_ind.columns = ['Industry', 'List Revenue ($)', 'Total Discount ($)', 'Net Revenue ($)', 'Discount Rate (%)']
display_disc_ind['List Revenue ($)']    = display_disc_ind['List Revenue ($)'].round(2)
display_disc_ind['Total Discount ($)'] = display_disc_ind['Total Discount ($)'].round(2)
display_disc_ind['Net Revenue ($)']    = display_disc_ind['Net Revenue ($)'].round(2)

print('Discount Rate by Industry (highest first):')
display_disc_ind

In [ ]:
# ── 4c: Discount by plan ──────────────────────────────────────────────────────
disc_by_plan = master.groupby('plan_name').agg(
    list_revenue  = ('base_revenue',         'sum'),
    total_disc    = ('discount',             'sum'),
    net_revenue   = ('subscription_revenue', 'sum'),
    list_price    = ('monthly_price',        'mean')
).reset_index()

disc_by_plan['discount_rate_pct'] = (disc_by_plan['total_disc'] / disc_by_plan['list_revenue'] * 100).round(2)

# Average realized price = list_price × (1 - discount_rate)
disc_by_plan['avg_realized_price'] = (
    disc_by_plan['list_price'] * (1 - disc_by_plan['discount_rate_pct'] / 100)
).round(2)

disc_by_plan = disc_by_plan.sort_values('discount_rate_pct', ascending=False).reset_index(drop=True)

display_disc_plan = disc_by_plan[['plan_name', 'list_price', 'avg_realized_price', 'total_disc', 'discount_rate_pct']].copy()
display_disc_plan.columns = ['Plan', 'List Price ($)', 'Avg Realized Price ($)', 'Total Discounts ($)', 'Discount Rate (%)']
display_disc_plan['Total Discounts ($)'] = display_disc_plan['Total Discounts ($)'].round(2)

print('Discount Rate by Plan (highest first):')
print('  → Avg Realized Price is the effective price after accounting for average discounts.')
print()
display_disc_plan

In [ ]:
# ── 4d: Discount flag — rows with zero discount vs discounted rows ─────────────
# We create a new column using numpy.where().
# np.where(condition, value_if_true, value_if_false) is like an IF statement.

master['has_discount'] = np.where(master['discount'] > 0, 'Discounted', 'No Discount')

disc_split = master.groupby('has_discount').agg(
    row_count     = ('customer_id',          'count'),
    total_revenue = ('subscription_revenue', 'sum')
).reset_index()

disc_split['pct_of_rows']    = (disc_split['row_count'] / disc_split['row_count'].sum() * 100).round(1)
disc_split['pct_of_revenue'] = (disc_split['total_revenue'] / disc_split['total_revenue'].sum() * 100).round(1)
disc_split['total_revenue']  = disc_split['total_revenue'].round(2)
disc_split.columns = ['Discount Status', 'Row Count', 'Total Revenue ($)', '% of Rows', '% of Revenue']
disc_split = disc_split.reset_index(drop=True)

print('Discounted vs Non-Discounted Revenue Rows:')
disc_split

**What does this tell us?**

- The overall discount rate shows how much revenue is being given away on average.
- Industry-level discount rates reveal which verticals receive the most preferential pricing.
- The realized price column shows the true effective price after discounts — and how far it deviates from list price.
- If a large proportion of transactions are discounted, the business is systematically collecting less than its listed rates.

---

## 5. Expansion Revenue Analysis

**Business question:** Are upsell opportunities being captured effectively?

**What is expansion revenue?**  
Expansion revenue is the incremental revenue earned from existing customers **on top of** their base subscription.  
This comes from add-ons, seat upgrades, or feature unlocks — it represents organic growth from the existing customer base.

**Why does it matter?**  
NovaSuite's target is for expansion revenue to make up **≥ 15%** of total revenue.  
Currently it sits at around **8%** — meaning the upsell motion is significantly under-leveraged.

A business that grows expansion revenue can increase total revenue even without acquiring new customers.  
This is the foundation of Net Revenue Retention (NRR) above 100%.

**What we will calculate:**
- Total expansion revenue and its share of overall revenue
- How often each plan generates expansion revenue (frequency of upsell events)
- Expansion revenue by company size
- Year-over-year expansion revenue trend

In [ ]:
# ── 5a: Overall expansion revenue summary ────────────────────────────────────
total_expansion  = master['expansion_revenue'].sum()
total_sub_rev    = master['subscription_revenue'].sum()
total_combined   = total_sub_rev + total_expansion
expansion_share  = (total_expansion / total_combined * 100).round(2)

# Count how many individual customer-month rows have any expansion revenue
rows_with_exp    = (master['expansion_revenue'] > 0).sum()
total_rows       = len(master)
pct_rows_with_exp = (rows_with_exp / total_rows * 100).round(1)

print('=' * 58)
print('  EXPANSION REVENUE OVERVIEW')
print('=' * 58)
print(f"  Total expansion revenue         : ${total_expansion:>10,.2f}")
print(f"  Total subscription revenue      : ${total_sub_rev:>10,.2f}")
print(f"  Expansion as % of all revenue   : {expansion_share:>10.2f}%")
print(f"  Target (from problem statement) :      15.00%")
print(f"  Gap to target                   : {15 - expansion_share:>10.2f}%")
print()
print(f"  Customer-months with expansion  : {rows_with_exp:,} of {total_rows:,} ({pct_rows_with_exp}%)")
print('=' * 58)

In [ ]:
# ── 5b: Expansion revenue by plan ─────────────────────────────────────────────
# We use a boolean condition to count how many rows have expansion_revenue > 0.
# Summing True/False values counts the True ones (True = 1, False = 0).

exp_by_plan = master.groupby('plan_name').agg(
    total_expansion   = ('expansion_revenue', 'sum'),
    total_rows        = ('expansion_revenue', 'count'),
    rows_with_exp     = ('expansion_revenue', lambda x: (x > 0).sum()),
    unique_customers  = ('customer_id',       'nunique')
).reset_index()

# Note: We use a named function here to count rows where expansion > 0.
# This could also be done in a separate step — see the step below for reference.

exp_by_plan['pct_months_with_exp'] = (
    exp_by_plan['rows_with_exp'] / exp_by_plan['total_rows'] * 100
).round(1)

exp_by_plan['avg_exp_per_customer'] = (
    exp_by_plan['total_expansion'] / exp_by_plan['unique_customers']
).round(2)

exp_by_plan = exp_by_plan.sort_values('total_expansion', ascending=False).reset_index(drop=True)

display_exp_plan = exp_by_plan[['plan_name', 'unique_customers', 'total_expansion', 'avg_exp_per_customer', 'pct_months_with_exp']].copy()
display_exp_plan.columns = ['Plan', 'Customers', 'Total Expansion Rev ($)', 'Avg Expansion / Customer ($)', '% Months with Expansion']
display_exp_plan['Total Expansion Rev ($)']          = display_exp_plan['Total Expansion Rev ($)'].round(2)
display_exp_plan['Avg Expansion / Customer ($)']     = display_exp_plan['Avg Expansion / Customer ($)'].round(2)

print('Expansion Revenue by Plan:')
print('  → % Months with Expansion = the share of customer-months where upsell occurred.')
print()
display_exp_plan

In [ ]:
# ── 5b (alternative — no lambda): count expansion rows using a separate step ──
# This approach avoids the lambda entirely and may be easier to read.

# Flag rows that have expansion revenue
master['has_expansion'] = master['expansion_revenue'] > 0   # Creates a True/False column

exp_check = master.groupby('plan_name').agg(
    rows_with_expansion = ('has_expansion', 'sum'),          # True counts as 1
    total_rows          = ('has_expansion', 'count')
).reset_index()

exp_check['pct_with_expansion'] = (exp_check['rows_with_expansion'] / exp_check['total_rows'] * 100).round(1)
exp_check.columns = ['Plan', 'Months with Expansion', 'Total Months', '% with Expansion']

print('Expansion frequency by plan (alternative calculation — same result, no lambda):')
exp_check

In [ ]:
# ── 5c: Expansion revenue by company size ─────────────────────────────────────
exp_by_size = master.groupby('company_size').agg(
    total_expansion  = ('expansion_revenue', 'sum'),
    unique_customers = ('customer_id',       'nunique')
).reset_index()

exp_by_size['avg_exp_per_customer'] = (
    exp_by_size['total_expansion'] / exp_by_size['unique_customers']
).round(2)

exp_by_size['pct_of_expansion'] = (
    exp_by_size['total_expansion'] / exp_by_size['total_expansion'].sum() * 100
).round(1)

exp_by_size = exp_by_size.sort_values('total_expansion', ascending=False).reset_index(drop=True)

display_exp_size = exp_by_size[['company_size', 'unique_customers', 'total_expansion', 'avg_exp_per_customer', 'pct_of_expansion']].copy()
display_exp_size.columns = ['Company Size', 'Customers', 'Total Expansion Rev ($)', 'Avg Expansion / Customer ($)', '% of All Expansion']
display_exp_size['Total Expansion Rev ($)']       = display_exp_size['Total Expansion Rev ($)'].round(2)
display_exp_size['Avg Expansion / Customer ($)']  = display_exp_size['Avg Expansion / Customer ($)'].round(2)

print('Expansion Revenue by Company Size:')
display_exp_size

In [ ]:
# ── 5d: Annual expansion revenue trend ───────────────────────────────────────
annual_expansion = master.groupby('year').agg(
    total_expansion = ('expansion_revenue',  'sum'),
    total_sub_rev   = ('subscription_revenue','sum')
).reset_index()

annual_expansion['expansion_share_pct'] = (
    annual_expansion['total_expansion']
    / (annual_expansion['total_expansion'] + annual_expansion['total_sub_rev'])
    * 100
).round(2)

display_exp_annual = annual_expansion[['year', 'total_expansion', 'total_sub_rev', 'expansion_share_pct']].copy()
display_exp_annual.columns = ['Year', 'Total Expansion Rev ($)', 'Subscription Rev ($)', 'Expansion Share (%)']
display_exp_annual['Total Expansion Rev ($)'] = display_exp_annual['Total Expansion Rev ($)'].round(2)
display_exp_annual['Subscription Rev ($)']    = display_exp_annual['Subscription Rev ($)'].round(2)
display_exp_annual = display_exp_annual.reset_index(drop=True)

print('Annual Expansion Revenue Trend:')
display_exp_annual

**What does this tell us?**

- If the percentage of months with expansion revenue is very low for Enterprise/Business plans, the upsell motion is not working consistently.
- Company size tells us which customer segments are growing their spend — and which are completely flat.
- The annual trend shows whether expansion revenue is growing as a share of total revenue, or staying stagnant.

---

## 6. ARPA Analysis (Average Revenue Per Account)

**Business question:** Which customers generate the highest average revenue?

**What is ARPA?**  
ARPA stands for **Average Revenue Per Account**.  
It tells us, on average, how much revenue one customer generates per month.

$$\text{ARPA} = \frac{\text{Total Monthly Revenue}}{\text{Number of Active Customers That Month}}$$

A rising ARPA means customers are spending more over time — either through price increases or upsells.  
A falling ARPA means the customer mix is shifting toward cheaper plans or discounts are increasing.

**NovaSuite KPI targets:**
- Current ARPA baseline: ~$95/month
- Target ARPA: ≥ $115/month

**What we will calculate:**
- Overall ARPA (monthly average)
- ARPA by plan
- ARPA by industry
- ARPA by company size
- Whether ARPA is improving year over year

In [ ]:
# ── 6a: Calculate overall monthly ARPA ───────────────────────────────────────
# ARPA is calculated per month: total revenue that month / unique customers that month.
# Then we take the average across all months to get the overall ARPA.

monthly_arpa = master.groupby('month').agg(
    monthly_revenue  = ('subscription_revenue', 'sum'),
    active_customers = ('customer_id',          'nunique')
).reset_index()

monthly_arpa['arpa'] = (monthly_arpa['monthly_revenue'] / monthly_arpa['active_customers']).round(2)

overall_arpa  = monthly_arpa['arpa'].mean().round(2)
arpa_target   = 115
arpa_gap      = (arpa_target - overall_arpa).round(2)

highest_arpa_row = monthly_arpa.loc[monthly_arpa['arpa'].idxmax()]
lowest_arpa_row  = monthly_arpa.loc[monthly_arpa['arpa'].idxmin()]

print('=' * 58)
print('  ARPA OVERVIEW')
print('=' * 58)
print(f"  Overall average ARPA     : ${overall_arpa:>8.2f}")
print(f"  Target ARPA              : ${arpa_target:>8.2f}")
print(f"  Gap to target            : ${arpa_gap:>8.2f}")
print()
print(f"  Highest monthly ARPA     : ${highest_arpa_row['arpa']:>8.2f}  ({highest_arpa_row['month'].strftime('%b %Y')})")
print(f"  Lowest  monthly ARPA     : ${lowest_arpa_row['arpa']:>8.2f}  ({lowest_arpa_row['month'].strftime('%b %Y')})")
print('=' * 58)

In [ ]:
# ── 6b: Annual ARPA trend ─────────────────────────────────────────────────────
# Group monthly ARPA values by year and average them.

monthly_arpa['year'] = monthly_arpa['month'].dt.year

annual_arpa = monthly_arpa.groupby('year').agg(
    avg_arpa         = ('arpa',             'mean'),
    avg_customers    = ('active_customers', 'mean')
).reset_index()

annual_arpa['avg_arpa']      = annual_arpa['avg_arpa'].round(2)
annual_arpa['avg_customers'] = annual_arpa['avg_customers'].round(0).astype(int)

# Flag whether ARPA is above or below target
annual_arpa['vs_target'] = np.where(
    annual_arpa['avg_arpa'] >= arpa_target,
    'Above target',
    'Below target'
)

display_annual_arpa = annual_arpa.copy()
display_annual_arpa.columns = ['Year', 'Avg ARPA ($)', 'Avg Active Customers', 'vs $115 Target']

print(f'Annual ARPA Trend (target = ${arpa_target}):')
display_annual_arpa

In [ ]:
# ── 6c: ARPA by plan ──────────────────────────────────────────────────────────
arpa_by_plan = master.groupby('plan_name').agg(
    total_rev        = ('subscription_revenue', 'sum'),
    unique_customers = ('customer_id',          'nunique'),
    list_price       = ('monthly_price',        'mean')
).reset_index()

arpa_by_plan['arpa'] = (arpa_by_plan['total_rev'] / arpa_by_plan['unique_customers']).round(2)

# Realisation rate: ARPA as % of list price (shows effective discount per plan)
arpa_by_plan['realisation_rate_pct'] = (arpa_by_plan['arpa'] / arpa_by_plan['list_price'] * 100).round(1)

arpa_by_plan = arpa_by_plan.sort_values('arpa', ascending=False).reset_index(drop=True)

display_arpa_plan = arpa_by_plan[['plan_name', 'unique_customers', 'list_price', 'arpa', 'realisation_rate_pct']].copy()
display_arpa_plan.columns = ['Plan', 'Customers', 'List Price ($)', 'ARPA ($)', 'Realisation Rate (%)']

print('ARPA by Plan:')
print('  → Realisation Rate = ARPA as % of list price. 100% = no discount, <100% = discounted.')
print()
display_arpa_plan

In [ ]:
# ── 6d: ARPA by industry ──────────────────────────────────────────────────────
arpa_by_industry = master.groupby('industry').agg(
    total_rev        = ('subscription_revenue', 'sum'),
    unique_customers = ('customer_id',          'nunique')
).reset_index()

arpa_by_industry['arpa'] = (arpa_by_industry['total_rev'] / arpa_by_industry['unique_customers']).round(2)
arpa_by_industry = arpa_by_industry.sort_values('arpa', ascending=False).reset_index(drop=True)

# Flag: above or below overall average ARPA
arpa_by_industry['vs_avg_arpa'] = np.where(
    arpa_by_industry['arpa'] >= overall_arpa,
    'Above average',
    'Below average'
)

display_arpa_ind = arpa_by_industry[['industry', 'unique_customers', 'total_rev', 'arpa', 'vs_avg_arpa']].copy()
display_arpa_ind.columns = ['Industry', 'Customers', 'Total Revenue ($)', 'ARPA ($)', f'vs Avg ARPA (${overall_arpa})']
display_arpa_ind['Total Revenue ($)'] = display_arpa_ind['Total Revenue ($)'].round(2)

print('ARPA by Industry (ranked highest to lowest):')
display_arpa_ind

In [ ]:
# ── 6e: ARPA by company size ──────────────────────────────────────────────────
arpa_by_size = master.groupby('company_size').agg(
    total_rev        = ('subscription_revenue', 'sum'),
    unique_customers = ('customer_id',          'nunique')
).reset_index()

arpa_by_size['arpa'] = (arpa_by_size['total_rev'] / arpa_by_size['unique_customers']).round(2)
arpa_by_size = arpa_by_size.sort_values('arpa', ascending=False).reset_index(drop=True)

display_arpa_size = arpa_by_size[['company_size', 'unique_customers', 'total_rev', 'arpa']].copy()
display_arpa_size.columns = ['Company Size', 'Customers', 'Total Revenue ($)', 'ARPA ($)']
display_arpa_size['Total Revenue ($)'] = display_arpa_size['Total Revenue ($)'].round(2)

print('ARPA by Company Size:')
display_arpa_size

**What does this tell us?**

- The realisation rate column shows how close each plan's effective price is to its list price — a rate below 100% means discounts are eroding ARPA.
- Industries above average ARPA are high-value accounts; industries below average need either upsell or pricing adjustments.
- Company size ARPA shows whether large companies are actually translating into higher per-account revenue.

---

## 7. Revenue Findings Summary

The following are factual observations drawn directly from the analysis above.  
These are **observations only** — strategic recommendations will be covered in the Pricing Strategy section.

---

### Finding 1 — MRR is Growing but Not Accelerating

Total MRR has increased year-over-year since January 2021, reflecting steady customer acquisition.  
However, the annual growth rate has not been consistent — average monthly MRR does not grow proportionally with the customer base.  
This indicates that new customers are being added at lower average price points, or existing discounts are widening over time.

---

### Finding 2 — Higher-Tier Plans Carry Most Revenue Despite Fewer Customers

Enterprise and Business plans contribute a disproportionately large share of total revenue relative to their customer count.  
The Basic plan has the most customers but the lowest total revenue contribution.  
This creates revenue concentration risk: losing a small number of Enterprise/Business accounts has an outsized impact on MRR.

---

### Finding 3 — Revenue Efficiency Varies Significantly by Plan

Revenue per user slot — total revenue divided by total available user capacity — varies across plans.  
Some plans generate high revenue per available seat; others have large user caps but low utilisation, meaning customers are paying for capacity they do not use.  
This signals a mismatch between plan structure and actual usage patterns.

---

### Finding 4 — Industry Revenue Is Concentrated in a Few Verticals

A small number of industries account for the majority of total revenue.  
The rank mismatch analysis reveals which industries have a large customer base but generate below-average revenue per account — these are under-monetised segments.  
EdTech and Retail rank lower on average revenue per customer relative to their customer count.

---

### Finding 5 — Discounts Are Applied Across All Segments

Discounts are not isolated to one plan or one industry — they appear across all tiers and verticals.  
The overall discount rate means realized revenue is measurably below list revenue across the entire customer base.  
EdTech and Retail have the highest discount rates by industry; Basic has the highest rate relative to its already low list price of $29.

---

### Finding 6 — Basic Plan Realised Price Falls Furthest from List Price

The Basic plan's average realised price is well below its $29 list price due to its high discount rate.  
At a $29 list price, even a modest percentage discount removes a significant portion of the revenue per account.  
This is the plan with the least room to absorb discounts — yet it appears to receive them frequently.

---

### Finding 7 — Expansion Revenue Is Significantly Below Target

Expansion revenue accounts for approximately 8% of total revenue — well below the 15% target.  
The percentage of customer-months that include any expansion revenue is low, indicating that upsell events are infrequent rather than systematic.  
Enterprise and Business plans generate the most expansion revenue in absolute terms, but the frequency of upsell in these accounts remains low.

---

### Finding 8 — Small Companies Generate Almost No Expansion Revenue

Small-sized companies contribute a negligible share of total expansion revenue.  
This is consistent with the pattern where Small customers on higher-tier plans (Business/Pro) use fewer features and are less likely to expand their spend.  
The expansion revenue opportunity is concentrated in Large and Enterprise customers.

---

### Finding 9 — Overall ARPA Is Below the $115 Target

Average revenue per account sits below the $115/month target across all years in the dataset.  
ARPA has not shown strong upward movement over time, suggesting that neither organic upsell nor price increases have materially lifted per-account revenue.  
The gap between current ARPA and the target represents a quantifiable revenue opportunity.

---

### Finding 10 — ARPA Varies Substantially Across Industries and Company Sizes

There is a meaningful spread in ARPA across industries — some verticals generate well above the overall average while others sit below it.  
Similarly, ARPA increases with company size, but the difference between Small and Enterprise accounts may be larger or smaller than the plan pricing suggests, due to discount variation.  
These ARPA gaps are an input for segment-specific pricing decisions in the next section.

---

